## Model Analysis (Using Outputs from Notebook 04)

This notebook focuses on:
- selecting an operational threshold
- error analysis by time + geography
- report-ready summary tables

#### Load up stored data

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS
from pyspark.ml.functions import vector_to_array

dbutils.widgets.dropdown("threshold_policy", "max_f1", ["max_f1", "recall_at_least_0_70"])
THRESHOLD_POLICY = dbutils.widgets.get("threshold_policy")
print("Threshold policy:", THRESHOLD_POLICY)

FINAL_SELECTION_TABLE = "workspace.bda_taxi.model_comparison_final"
BEST_PRED_TABLE = "workspace.bda_taxi.model_preds_best"
THRESHOLD_TABLE = "workspace.bda_taxi.model_threshold_metrics"

final_selection = spark.table(FINAL_SELECTION_TABLE)
best_preds = spark.table(BEST_PRED_TABLE)

display(final_selection)
print("Best predictions rows:", best_preds.count())
display(best_preds.limit(5))

#### Threshold selection

Initially choosing the threshold that maximises F1 for the best model

In [0]:
threshold_metrics = spark.table(THRESHOLD_TABLE)
display(threshold_metrics.orderBy("threshold", "model_name"))

# Choose the threshold
best_model_name = final_selection.select("model").first()["model"]

best_threshold_row = (
    threshold_metrics
    .filter(SQL_FUNCTIONS.col("model_name") == best_model_name)
    .orderBy(SQL_FUNCTIONS.desc("f1_at_threshold"))
    .limit(1)
    .collect()[0]
)

CHOSEN_THRESHOLD = float(best_threshold_row["threshold"])
print("Chosen threshold:", CHOSEN_THRESHOLD, "for model:", best_model_name)
print("Row:", best_threshold_row)

In [0]:
scored = (
    best_preds
    .withColumn("prob_array", vector_to_array(SQL_FUNCTIONS.col("probability")))
    .withColumn("p_tip", SQL_FUNCTIONS.col("prob_array")[1].cast("double"))
    .withColumn("label_int", SQL_FUNCTIONS.col("label").cast("int"))
    .withColumn("pred_at_threshold", SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("p_tip") >= CHOSEN_THRESHOLD, 1).otherwise(0))
    .withColumn("is_fp", SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 1) & (SQL_FUNCTIONS.col("label_int") == 0), 1).otherwise(0))
    .withColumn("is_fn", SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 0) & (SQL_FUNCTIONS.col("label_int") == 1), 1).otherwise(0))
    .withColumn("is_error", SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("pred_at_threshold") != SQL_FUNCTIONS.col("label_int"), 1).otherwise(0))
)

scored.selectExpr(
    "count(*) as rows",
    "avg(is_error) as error_rate",
    "sum(is_fp) as false_positives",
    "sum(is_fn) as false_negatives"
).show(truncate=False)

In [0]:
## Best model performance at chosen threshold (report summary)

threshold_perf = (
    scored
    .agg(
        SQL_FUNCTIONS.count("*").alias("trip_count"),
        SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 1) & (SQL_FUNCTIONS.col("label_int") == 1), 1).otherwise(0)).alias("tp"),
        SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 1) & (SQL_FUNCTIONS.col("label_int") == 0), 1).otherwise(0)).alias("fp"),
        SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 0) & (SQL_FUNCTIONS.col("label_int") == 1), 1).otherwise(0)).alias("fn"),
        SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 0) & (SQL_FUNCTIONS.col("label_int") == 0), 1).otherwise(0)).alias("tn")
    )
    .withColumn("threshold", SQL_FUNCTIONS.lit(float(CHOSEN_THRESHOLD)))
    .withColumn("precision", SQL_FUNCTIONS.round(SQL_FUNCTIONS.col("tp") / (SQL_FUNCTIONS.col("tp") + SQL_FUNCTIONS.col("fp")), 4))
    .withColumn("recall", SQL_FUNCTIONS.round(SQL_FUNCTIONS.col("tp") / (SQL_FUNCTIONS.col("tp") + SQL_FUNCTIONS.col("fn")), 4))
    .withColumn(
        "f1",
        SQL_FUNCTIONS.round(
            2 * (SQL_FUNCTIONS.col("precision") * SQL_FUNCTIONS.col("recall")) /
            (SQL_FUNCTIONS.col("precision") + SQL_FUNCTIONS.col("recall")),
            4
        )
    )
    .withColumn("accuracy", SQL_FUNCTIONS.round((SQL_FUNCTIONS.col("tp") + SQL_FUNCTIONS.col("tn")) / SQL_FUNCTIONS.col("trip_count"), 4))
)

display(threshold_perf)

#### Calibration check (probability vs actual tip rate)

In [0]:

# Create bins of predicted probability and compare predicted vs actual
calibration = (
    scored
    .withColumn("p_bin", (SQL_FUNCTIONS.floor(SQL_FUNCTIONS.col("p_tip") * 10) / 10).cast("double"))
    .groupBy("p_bin")
    .agg(
        SQL_FUNCTIONS.count("*").alias("trip_count"),
        SQL_FUNCTIONS.avg("p_tip").alias("avg_predicted_prob"),
        SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate")
    )
    .orderBy("p_bin")
)

display(calibration)

CALIBRATION_TABLE = "workspace.bda_taxi.best_model_calibration_bins"
(
    calibration.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(CALIBRATION_TABLE)
)
print("Saved calibration table:", CALIBRATION_TABLE)

#### Store threshold selected

In [0]:
THRESHOLD_CHOICE_TABLE = "workspace.bda_taxi.best_model_threshold_choice"

threshold_choice_df = spark.createDataFrame(
    [(best_model_name, float(CHOSEN_THRESHOLD))],
    ["model_name", "chosen_threshold"]
).withColumn("chosen_ts", SQL_FUNCTIONS.current_timestamp())

(
    threshold_choice_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(THRESHOLD_CHOICE_TABLE)
)

print("Saved threshold choice:", THRESHOLD_CHOICE_TABLE)
display(spark.table(THRESHOLD_CHOICE_TABLE))

### Error analysis

#### Error analysis by time

In [0]:
# By time_bucket
if "time_bucket" in scored.columns:
    by_bucket = (
        scored.groupBy("time_bucket")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )
    display(by_bucket)

# By pickup_hour
if "pickup_hour" in scored.columns:
    by_hour = (
        scored.groupBy("pickup_hour")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy("pickup_hour")
    )
    display(by_hour)

#### Error analysis by geography

In [0]:
# Pickup zones
if "PU_Zone" in scored.columns:
    by_pu_zone = (
        scored.groupBy("PU_Zone")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )
    display(by_pu_zone.limit(30))

# Dropoff zones
if "DO_Zone" in scored.columns:
    by_do_zone = (
        scored.groupBy("DO_Zone")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("label_int").cast("double")).alias("actual_tip_rate"),
            SQL_FUNCTIONS.avg(SQL_FUNCTIONS.col("pred_at_threshold").cast("double")).alias("predicted_tip_rate"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )
    display(by_do_zone.limit(30))

#### False positive vs False negative concentration

Using a minimum number of trips set at 200 to ensure there is no impact from outliers (could see in notebook 03 in the visualisations the outliers)

In [0]:

def error_slices_by_column(scored_df, group_col: str, top_n: int = 25):
    return (
        scored_df.groupBy(group_col)
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.sum("is_fp").alias("false_positives"),
            SQL_FUNCTIONS.sum("is_fn").alias("false_negatives"),
            SQL_FUNCTIONS.avg("is_error").alias("error_rate")
        )
        .withColumn("fp_rate", SQL_FUNCTIONS.col("false_positives") / SQL_FUNCTIONS.col("trip_count"))
        .withColumn("fn_rate", SQL_FUNCTIONS.col("false_negatives") / SQL_FUNCTIONS.col("trip_count"))
        .orderBy(SQL_FUNCTIONS.desc("trip_count"))
    )

# Top FN rate zones (with minimum volume filter for fairness)
MIN_TRIPS = 200

if "PU_Zone" in scored.columns:
    pu_zone_slices = error_slices_by_column(scored, "PU_Zone")
    display(
        pu_zone_slices
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .orderBy(SQL_FUNCTIONS.desc("fn_rate"))
        .limit(25)
    )

if "DO_Zone" in scored.columns:
    do_zone_slices = error_slices_by_column(scored, "DO_Zone")
    display(
        do_zone_slices
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .orderBy(SQL_FUNCTIONS.desc("fn_rate"))
        .limit(25)
    )

##### Store the above False Postive and False Negative results

In [0]:
FPFN_PU_ZONE_TABLE = "workspace.bda_taxi.best_model_fpfn_by_pu_zone"
FPFN_DO_ZONE_TABLE = "workspace.bda_taxi.best_model_fpfn_by_do_zone"

if "PU_Zone" in scored.columns:
    (
        pu_zone_slices
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(FPFN_PU_ZONE_TABLE)
    )
    print("Saved:", FPFN_PU_ZONE_TABLE)

if "DO_Zone" in scored.columns:
    (
        do_zone_slices
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(FPFN_DO_ZONE_TABLE)
    )
    print("Saved:", FPFN_DO_ZONE_TABLE)

#### Store data to be used for report

In [0]:
OUTPUT_SCORED_TABLE = "workspace.bda_taxi.best_model_scored"
OUTPUT_ERROR_BUCKET_TABLE = "workspace.bda_taxi.best_model_error_by_time_bucket"
ERROR_BY_HOUR_TABLE = "workspace.bda_taxi.best_model_error_by_pickup_hour"
ERROR_BY_PU_ZONE_TABLE = "workspace.bda_taxi.best_model_error_by_pu_zone"
ERROR_BY_DO_ZONE_TABLE = "workspace.bda_taxi.best_model_error_by_do_zone"

(
    scored.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(OUTPUT_SCORED_TABLE)
)
print("Saved:", OUTPUT_SCORED_TABLE)

if "time_bucket" in scored.columns:
    (
        by_bucket.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(OUTPUT_ERROR_BUCKET_TABLE)
    )
    print("Saved:", OUTPUT_ERROR_BUCKET_TABLE)

if "pickup_hour" in scored.columns:
    (
        by_hour.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(ERROR_BY_HOUR_TABLE)
    )
    print("Saved:", ERROR_BY_HOUR_TABLE)

if "PU_Zone" in scored.columns:
    (
        by_pu_zone.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(ERROR_BY_PU_ZONE_TABLE)
    )
    print("Saved:", ERROR_BY_PU_ZONE_TABLE)

if "DO_Zone" in scored.columns:
    (
        by_do_zone.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(ERROR_BY_DO_ZONE_TABLE)
    )
    print("Saved:", ERROR_BY_DO_ZONE_TABLE)

##### Confirming outputs (sanity check)

In [0]:

print("Scored table rows:", spark.table(OUTPUT_SCORED_TABLE).count())
print("Calibration bins rows:", spark.table(CALIBRATION_TABLE).count())
print("Threshold choice rows:", spark.table(THRESHOLD_CHOICE_TABLE).count())

spark.table(OUTPUT_SCORED_TABLE).selectExpr(
    "min(p_tip) as min_p_tip",
    "max(p_tip) as max_p_tip",
    "avg(p_tip) as avg_p_tip",
    "avg(label_int) as actual_tip_rate"
).show(truncate=False)

In [0]:
# Key results for technical report (data from saved tables)

# Core outputs from Notebook 04 + Notebook 05
MODEL_SELECTION_TABLE = "workspace.bda_taxi.model_comparison_final"
THRESHOLD_CHOICE_TABLE = "workspace.bda_taxi.best_model_threshold_choice"
THRESHOLD_METRICS_TABLE = "workspace.bda_taxi.model_threshold_metrics"
SCORED_TABLE = "workspace.bda_taxi.best_model_scored"
CALIBRATION_TABLE = "workspace.bda_taxi.best_model_calibration_bins"

ERROR_BY_BUCKET_TABLE = "workspace.bda_taxi.best_model_error_by_time_bucket"
ERROR_BY_HOUR_TABLE = "workspace.bda_taxi.best_model_error_by_pickup_hour"
ERROR_BY_PU_ZONE_TABLE = "workspace.bda_taxi.best_model_error_by_pu_zone"
ERROR_BY_DO_ZONE_TABLE = "workspace.bda_taxi.best_model_error_by_do_zone"
FPFN_PU_ZONE_TABLE = "workspace.bda_taxi.best_model_fpfn_by_pu_zone"
FPFN_DO_ZONE_TABLE = "workspace.bda_taxi.best_model_fpfn_by_do_zone"

# Interpretation (from 05a interpretation-only notebook)
LR_TOP_POS = "workspace.bda_taxi.logreg_coefficients_top_positive"
LR_TOP_NEG = "workspace.bda_taxi.logreg_coefficients_top_negative"

# Optional regression (05b)
REG_METRICS_TABLE = "workspace.bda_taxi.fare_regression_metrics"

print("=== Best model selection ===")
display(spark.table(MODEL_SELECTION_TABLE))

print("=== Chosen threshold ===")
display(spark.table(THRESHOLD_CHOICE_TABLE))

print("=== Performance vs thresholds (winner model only) ===")
winner = spark.table(MODEL_SELECTION_TABLE).select("model").first()["model"]
display(
    spark.table(THRESHOLD_METRICS_TABLE)
    .filter(SQL_FUNCTIONS.col("model_name") == winner)
    .orderBy("threshold")
)

print("=== Performance summary at chosen threshold ===")
scored = spark.table(SCORED_TABLE)
scored.selectExpr(
    "count(*) as trip_count",
    "avg(label_int) as actual_tip_rate",
    "avg(pred_at_threshold) as predicted_tip_rate",
    "avg(is_error) as error_rate",
    "sum(is_fp) as false_positives",
    "sum(is_fn) as false_negatives"
).show(truncate=False)

print("=== Calibration bins (pred prob vs actual) ===")
display(spark.table(CALIBRATION_TABLE).orderBy("p_bin"))

print("=== Error analysis by time bucket ===")
display(spark.table(ERROR_BY_BUCKET_TABLE).orderBy(SQL_FUNCTIONS.desc("trip_count")))

print("=== Error analysis by pickup hour (top trip_count) ===")
display(spark.table(ERROR_BY_HOUR_TABLE).orderBy(SQL_FUNCTIONS.desc("trip_count")).limit(24))

print("=== Error analysis by PU/DO zones (top trip_count) ===")
display(spark.table(ERROR_BY_PU_ZONE_TABLE).orderBy(SQL_FUNCTIONS.desc("trip_count")).limit(25))
display(spark.table(ERROR_BY_DO_ZONE_TABLE).orderBy(SQL_FUNCTIONS.desc("trip_count")).limit(25))

print("=== Where the model fails (FP/FN concentration) ===")
display(spark.table(FPFN_PU_ZONE_TABLE).orderBy(SQL_FUNCTIONS.desc("fn_rate")).limit(25))
display(spark.table(FPFN_DO_ZONE_TABLE).orderBy(SQL_FUNCTIONS.desc("fn_rate")).limit(25))

print("=== Logistic Regression interpretation (top drivers) ===")
if winner == "LogisticRegression":
    display(spark.table(LR_TOP_POS).orderBy(SQL_FUNCTIONS.desc("coef")).limit(20))
    display(spark.table(LR_TOP_NEG).orderBy(SQL_FUNCTIONS.asc("coef")).limit(20))
else:
    print("Winner is not LogisticRegression, skip LR coefficient tables.")

print("=== Supporting regression results (fare prediction) ===")
display(spark.table(REG_METRICS_TABLE).orderBy("rmse"))

## Conclusions (final analysis layer)

### Model selection and decision policy
- The selected classification model is taken from **`workspace.bda_taxi.model_comparison_final`** and was chosen based on the best AUC-ROC among the evaluated supervised models (Logistic Regression vs Random Forest).
- An operational decision threshold was selected using the stored threshold sweep (**`workspace.bda_taxi.model_threshold_metrics`**) and persisted in **`workspace.bda_taxi.best_model_threshold_choice`**.  
- This threshold determines how predicted probabilities are converted into a final tipping prediction (tip/no-tip) and provides a defensible basis for reporting precision/recall/F1 trade-offs.

### Model reliability and calibration
- Calibration results in **`workspace.bda_taxi.best_model_calibration_bins`** compare predicted probabilities to observed tip rates across probability bands.
- A well-calibrated model should show that higher predicted probabilities correspond to higher observed tip rates (useful for explaining model trustworthiness in the report).

### Where the model performs well vs poorly (error analysis)
- The scored dataset **`workspace.bda_taxi.best_model_scored`** includes:  
`p_tip` (predicted probability), `pred_at_threshold` (decision), and error flags (`is_fp`, `is_fn`, `is_error`).
- Error slices by time show how performance varies over time:
  - **Time bucket:** `workspace.bda_taxi.best_model_error_by_time_bucket`
  - **Pickup hour:** `workspace.bda_taxi.best_model_error_by_pickup_hour`
- Error slices by location show spatial patterns and highlight where the model under/over-predicts tipping:
  - **Pickup zone:** `workspace.bda_taxi.best_model_error_by_pu_zone`
  - **Dropoff zone:** `workspace.bda_taxi.best_model_error_by_do_zone`
  - Concentration of **false positives vs false negatives** is captured in:
    - `workspace.bda_taxi.best_model_fpfn_by_pu_zone`
    - `workspace.bda_taxi.best_model_fpfn_by_do_zone`

### Interpretation of tipping behaviour
- **Proposal Q1 (tip likelihood + influential trip characteristics):**  
  - Model interpretation is provided via coefficient tables (Logistic Regression) from the interpretation-only notebook:
    - `workspace.bda_taxi.logreg_coefficients_top_positive`
    - `workspace.bda_taxi.logreg_coefficients_top_negative`  
    - These highlight which features most increase/decrease tipping probability.
- **Proposal Q2 & Q3 (spatial variation in fares and tips):**  
  - Zone-level performance and observed/predicted tipping rates support spatial conclusions (pickup/dropoff zones).
- **Proposal Q4 (three 8-hour intervals):**  
  - The `time_bucket` slices and observed/predicted tip rates directly support the time-interval analysis.

### Supporting predictive analysis (fare regression)
- The proposal’s regression extension is completed in Notebook 05_model_interpretation_error_analysis_and_final_outputs_fare_regression, with results stored in:
  - `workspace.bda_taxi.fare_regression_metrics`
  - Linear Regression serves as the baseline, and Decision Tree Regressor captures non-linear effects. Performance is assessed using RMSE and R².

### Limitations and assumptions
- Tips are only reliably observable for **credit card trips**, so modelling and evaluation focus on the subset where tipping is recorded.
- Withinn the United States, tipping is considered part of the culture for most service work, so this has to be considerd when looking at the high percentage of tips.
- Zone-level conclusions should be interpreted alongside trip counts, since low-volume zones can produce unstable averages.
- This project is based on a limited time window dataset; patterns may differ across seasons or broader time ranges.